In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
train_path = "../dataset/Training"
test_path = "../dataset/Testing"

print("Training path:", train_path)
print("Testing path:", test_path)

Training path: ../dataset/Training
Testing path: ../dataset/Testing


In [5]:
train_path = "../dataset/Training"
test_path = "../dataset/Testing"

import os

print("Training folder exists:", os.path.exists(train_path))
print("Testing folder exists:", os.path.exists(test_path))

Training folder exists: True
Testing folder exists: True


In [6]:
import os

train_path = r"D:\brain-tumor-mlops\dataset\Training"
test_path = r"D:\brain-tumor-mlops\dataset\Testing"

def count_images(folder):
    counts = {}
    for category in os.listdir(folder):
        category_path = os.path.join(folder, category)

        if os.path.isdir(category_path):
            counts[category] = len(os.listdir(category_path))

    return counts


print("Training dataset:")
print(count_images(train_path))

print("\nTesting dataset:")
print(count_images(test_path))

Training dataset:
{'glioma': 1400, 'meningioma': 1400, 'notumor': 1400, 'pituitary': 1400}

Testing dataset:
{'glioma': 400, 'meningioma': 400, 'notumor': 400, 'pituitary': 400}


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_path = r"D:\brain-tumor-mlops\dataset\Training"
test_path = r"D:\brain-tumor-mlops\dataset\Testing"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training + validation preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Testing preprocessing
test_datagen = ImageDataGenerator(
    rescale=1./255
)

# Training data
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

# Validation data
validation_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Testing data
test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.


In [8]:
print(train_generator.class_indices)

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models

NUM_CLASSES = 4

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES, activation='softmax')
])


model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 26, 26, 128)       0

In [10]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

Epoch 1/10
140/140 [==============================] - 282s 2s/step - loss: 0.7751 - accuracy: 0.6815 - val_loss: 0.5042 - val_accuracy: 0.8170
Epoch 2/10
140/140 [==============================] - 257s 2s/step - loss: 0.4278 - accuracy: 0.8397 - val_loss: 0.3789 - val_accuracy: 0.8634
Epoch 3/10
140/140 [==============================] - 244s 2s/step - loss: 0.3116 - accuracy: 0.8929 - val_loss: 0.3077 - val_accuracy: 0.8920
Epoch 4/10
140/140 [==============================] - 252s 2s/step - loss: 0.2214 - accuracy: 0.9199 - val_loss: 0.2859 - val_accuracy: 0.8929
Epoch 5/10
140/140 [==============================] - 249s 2s/step - loss: 0.1664 - accuracy: 0.9400 - val_loss: 0.2897 - val_accuracy: 0.9098
Epoch 6/10
140/140 [==============================] - 248s 2s/step - loss: 0.1374 - accuracy: 0.9500 - val_loss: 0.2325 - val_accuracy: 0.9268
Epoch 7/10
140/140 [==============================] - 282s 2s/step - loss: 0.0950 - accuracy: 0.9676 - val_loss: 0.2476 - val_accuracy: 0.9250

In [12]:
model.save("brain_tumor_cnn_baseline.h5")

d:\brain-tumor-mlops\venv\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
model.save("brain_tumor_cnn_baseline.keras")

In [14]:
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

50/50 [==============================] - 42s 849ms/step - loss: 1.2043 - accuracy: 0.8694
Test Loss: 1.2042968273162842
Test Accuracy: 0.8693749904632568


In [15]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Predictions
predictions = model.predict(test_generator)

# Convert probabilities to class labels
predicted_classes = np.argmax(predictions, axis=1)

# True labels
true_classes = test_generator.classes

# Class names
class_names = list(test_generator.class_indices.keys())

# Classification report
print(classification_report(
    true_classes,
    predicted_classes,
    target_names=class_names
))

50/50 [==============================] - 23s 449ms/step
              precision    recall  f1-score   support

      glioma       0.92      0.71      0.80       400
  meningioma       0.79      0.86      0.82       400
     notumor       0.85      0.99      0.91       400
   pituitary       0.95      0.92      0.93       400

    accuracy                           0.87      1600
   macro avg       0.88      0.87      0.87      1600
weighted avg       0.88      0.87      0.87      1600

